# LEVIR-CD: Building Change Detection với U-Net

Notebook chạy trên Kaggle GPU. Trước khi Run All: **Add Input** → dataset `mdrifaturrahman33/levir-cd`; bật **Accelerator: GPU** và **Internet: On**. Dùng cho học thuật theo điều khoản của tác giả LEVIR-CD.

Quy trình: kiểm tra dữ liệu → baseline RGB difference → train early-fusion U-Net → chọn ngưỡng trên validation → IoU/F1 trên test → hình so sánh. Siamese U-Net là phần so sánh tùy chọn. Không dùng số từ smoke test cho CV.

In [ ]:
from pathlib import Path
import json, subprocess, sys, torch
from IPython.display import display, Markdown, Image

assert torch.cuda.is_available(), 'Hãy bật GPU trong Kaggle Settings rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
REPO = Path('/kaggle/working/satellite-building-change-detection')
REVISION = 'b2eb1b9'  # Mã nguồn đã kiểm tra; thay bằng commit mới nếu muốn cập nhật
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LQTYulJinn/satellite-building-change-detection.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', REVISION], cwd=REPO, check=True)
print('Source:', REPO, '| commit:', REVISION)

In [ ]:
input_root = Path('/kaggle/input')
matches = [p.parent for p in input_root.rglob('train')
           if p.is_dir() and (p / 'A').is_dir()
           and (p.parent / 'val' / 'A').is_dir()
           and (p.parent / 'test' / 'A').is_dir()]
assert len(matches) == 1, f'Cần đúng một bộ LEVIR-CD. Tìm thấy: {matches}. Kiểm tra Add Input.'
DATA = matches[0]
print('Dataset:', DATA)
def run(script, *args):
    command = [sys.executable, '-u', str(REPO / script), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)
run('audit_data.py', '--data', DATA)

## Baseline
Chọn ngưỡng RGB difference trên validation. Kết quả test được tính một lần sau khi chọn ngưỡng.

In [ ]:
run('baseline.py', '--data', DATA)

## Early-fusion U-Net
Input gồm ảnh trước và sau (6 kênh RGB). Loss = BCEWithLogits + Dice để học pixel thay đổi hiếm gặp. Checkpoint tốt nhất được chọn theo validation F1 ở ngưỡng 0.5.

Nếu Kaggle báo hết VRAM, giảm `BATCH_SIZE` xuống 4 và chạy lại từ cell này; ghi lại cấu hình thực dùng.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 8
BASE_CHANNELS = 32
SEED = 42
run('train.py', '--data', DATA, '--run', 'runs/early', '--architecture', 'early',
    '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--base', BASE_CHANNELS,
    '--workers', 2, '--seed', SEED)
run('evaluate.py', '--data', DATA, '--checkpoint', 'runs/early/best.pt', '--split', 'test')

## Tùy chọn: Siamese U-Net
Chạy cell này nếu còn quota GPU để so sánh kiến trúc trên cùng split, seed và cấu hình. Có thể bỏ qua và vẫn có kết quả early-fusion.

In [ ]:
RUN_SIAMESE = False  # đổi thành True để chạy mô hình thứ hai
if RUN_SIAMESE:
    run('train.py', '--data', DATA, '--run', 'runs/siamese', '--architecture', 'siamese',
        '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--base', BASE_CHANNELS,
        '--workers', 2, '--seed', SEED)
    run('evaluate.py', '--data', DATA, '--checkpoint', 'runs/siamese/best.pt', '--split', 'test')

## Kết quả và hình so sánh
IoU/F1 được tính từ tổng TP/FP/FN của mọi pixel test. Ngưỡng dự đoán được chọn trên validation, không chọn trên test. Hình gồm ảnh trước, sau, ground truth và dự đoán.

In [ ]:
run('summarize.py')
display(Markdown((REPO / 'runs/results.md').read_text(encoding='utf-8')))
metrics = json.loads((REPO / 'runs/early/test/metrics.json').read_text(encoding='utf-8'))
print('Early-fusion test:', json.dumps(metrics['metrics'], indent=2))
figures = sorted((REPO / 'runs/early/test/examples').glob('*.png'))
for figure in figures[:4]:
    print(figure.name)
    display(Image(filename=str(figure)))

## Xuất kết quả
Tải `levir_cd_results.zip` từ Output của notebook. File gồm audit, metrics, cấu hình, checkpoint và hình; không đưa checkpoint hoặc dữ liệu vào Git thường. Chỉ ghi số lên CV sau khi notebook hoàn tất.

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/levir_cd_results', 'zip', root_dir=REPO, base_dir='runs')
print('Download:', archive)